In [1]:
import pandas as pd
import numpy as np
import datetime
import GHEtool as ghe
import json
from matplotlib import pyplot as plt
import sys
module_path = r"C:\Workdir\Develop\IOCS"
if module_path not in sys.path:
    sys.path.append(module_path)
from COP_tables import COP
import load_params
from borefield_modeling import Borefield
import TACO_functions as uf

In [ ]:
path_outputsAll = r"C:\Users\u0148284\OneDrive - KU Leuven\Taco\ResultFiles\IOCS\25250415_ResultsAfterBorefieldFix\StijnStreuvel_TMY\Iteration5\outputsAll.csv"
path_OutputNames = r"C:\Users\u0148284\OneDrive - KU Leuven\Taco\ResultFiles\IOCS\25250415_ResultsAfterBorefieldFix\StijnStreuvel_TMY\Iteration5\OutputNames.txt"
df = uf.read_ocp_result(path_outputsAll, path_OutputNames)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\u0148284\\OneDrive - KU Leuven\\Taco\\ResultFiles\\IOCS\\25250415_ResultsAfterBorefieldFix\\StijnStreuvel\\Iteration5\\OutputNames.txt'

In [ ]:
%matplotlib qt
QBor = df['QBor'].to_numpy()/1e3
bor_extraction_TACO = np.where(QBor < 0, -QBor, 0)
bor_injection_TACO = np.where(QBor > 0, QBor, 0)

path_result_dict = r"C:\Users\u0148284\OneDrive - KU Leuven\Taco\ResultFiles\IOCS\StijnStreuvel_wComf10000_TMY\Iteration5\sizing_result_dict.json"
with open(path_result_dict, 'r') as f:
    result_dict = json.load(f)

bor_injection_SizeOpt = np.array(result_dict["Borefield"]["cool"]) + np.array(result_dict["Borefield"]["reg"])
bor_extraction_SizeOpt = np.array(result_dict["Borefield"]["heat"])

plt.figure("compare loads")
plt.plot(bor_extraction_TACO, label="TACO extraction", color='blue')
plt.plot(bor_extraction_SizeOpt, label="SizeOpt extraction", color='blue', linestyle='dashed')
plt.plot(bor_injection_TACO, label="TACO injection", color='orange')
plt.plot(bor_injection_SizeOpt, label="SizeOpt injection", color='orange', linestyle='dashed')
plt.xlabel("Time step")
plt.ylabel("Borefield load [kW]")
plt.legend(), plt.grid()
plt.show()

In [ ]:
## Set the borefield parameters
# Load parameters
param, devs, dem, result_dict = load_params.load_params()


In [ ]:
## Size with TACO load
borefield = ghe.Borefield()

# set ground data in borefield
borefield.set_ground_parameters(devs["Borefield"]["bor_params"]["ground_data"])
borefield.set_fluid_parameters(devs["Borefield"]["bor_params"]["fluid_data"])
borefield.set_pipe_parameters(devs["Borefield"]["bor_params"]["pipe_data"])

borefield.create_rectangular_borefield(N_1=devs["Borefield"]["bor_params"]["N_1"], N_2=devs["Borefield"]["bor_params"]["N_2"], B_1=devs["Borefield"]["bor_params"]["B"], B_2=devs["Borefield"]["bor_params"]["B"], \
                                            H=100, D=devs["Borefield"]["bor_params"]["D"], r_b=devs["Borefield"]["bor_params"]["r_b"])

# set temperature bounds
borefield.set_max_avg_fluid_temperature(devs["Borefield"]["bor_params"]["Tf_max"])
borefield.set_min_avg_fluid_temperature(devs["Borefield"]["bor_params"]["Tf_min"])

# load the hourly profile
borefield.simulation_period = devs["Borefield"]["life_time"]
load = ghe.HourlyGeothermalLoad(simulation_period=devs["Borefield"]["life_time"])

load.hourly_extraction_load = bor_extraction_TACO
load.hourly_injection_load = bor_injection_TACO
borefield.load = load
load.all_months_equal = True
load.peak_extraction_duration = 10

colors = plt.cm.tab10(np.linspace(0, 1, 10))
Tb_GHE_monthly = {}
Tf_GHE_monthly = {}
Tb_GHE_hourly_ST = {}
Tf_GHE_hourly_ST = {}

## options without short-term effects
options = {"disp": False, "profiles": True, "method": "equivalent"}
## set options
borefield.set_options_gfunction_calculation(options)


L3 = borefield.size_L3()
print("L3: ", L3)
borefield.print_temperature_profile()
L4 = borefield.size_L4()
borefield.print_temperature_profile()
print("L4: ", L4)
print("Yearly injection: ", np.sum(bor_injection_TACO)/1e3, "MWh")
print("Yearly extraction: ", np.sum(bor_extraction_TACO)/1e3, "MWh")
print("Yearly imbalance: ", (np.sum(bor_injection_TACO) - np.sum(bor_extraction_TACO))/1e3, "MWh")


In [ ]:
## Size with Sizeopt load


# initiate borefield
borefield = ghe.Borefield()

# set ground data in borefield
borefield.set_ground_parameters(devs["Borefield"]["bor_params"]["ground_data"])
borefield.set_fluid_parameters(devs["Borefield"]["bor_params"]["fluid_data"])
borefield.set_pipe_parameters(devs["Borefield"]["bor_params"]["pipe_data"])

borefield.create_rectangular_borefield(N_1=devs["Borefield"]["bor_params"]["N_1"], N_2=devs["Borefield"]["bor_params"]["N_2"], B_1=devs["Borefield"]["bor_params"]["B"], B_2=devs["Borefield"]["bor_params"]["B"], \
                                            H=100, D=devs["Borefield"]["bor_params"]["D"], r_b=devs["Borefield"]["bor_params"]["r_b"])

# set temperature bounds
borefield.set_max_avg_fluid_temperature(devs["Borefield"]["bor_params"]["Tf_max"])
borefield.set_min_avg_fluid_temperature(devs["Borefield"]["bor_params"]["Tf_min"])

# load the hourly profile
borefield.simulation_period = devs["Borefield"]["life_time"]
load = ghe.HourlyGeothermalLoad(simulation_period=devs["Borefield"]["life_time"])

load.hourly_extraction_load = bor_extraction_SizeOpt
load.hourly_injection_load = bor_injection_SizeOpt
borefield.load = load
load.all_months_equal = True
peak_ext = np.max(bor_extraction_SizeOpt)

peak_locations = np.where(bor_extraction_SizeOpt >= 0.5*peak_ext, True, False)
# Determine the maximum amount of consecutive True values in peak_locations
max_consecutive_true = 0
current_count = 0

for value in peak_locations:
    if value:
        current_count += 1
        max_consecutive_true = max(max_consecutive_true, current_count)
    else:
        current_count = 0

print("Maximum consecutive True values:", max_consecutive_true)

load.peak_extraction_duration = max_consecutive_true





# def determine_peak_duration(inj_load, ext_load):
#     """
#     Function to determine the peak duration of the load profile
#     :param inj_load: injection load profile
#     :param ext_load: extraction load profile
#     :return:
#     """
#     max_inj = np.max(inj_load)
#     max_ext = np.max(ext_load)

#     inj_durations = np.diff(np.where(np.concatenate(([False], inj_load == max_inj, [False])))[0])
#     ext_durations = np.diff(np.where(np.concatenate(([False], ext_load == max_ext, [False])))[0])

#     max_inj_duration = np.max(inj_durations) if len(inj_durations) > 0 else 0
#     max_ext_duration = np.max(ext_durations) if len(ext_durations) > 0 else 0

#     if max_inj_duration >= max_ext_duration:
#         return np.argmax(inj_durations) if len(inj_durations) > 0 else -1
#     else:
#         return np.argmax(ext_durations) if len(ext_durations) > 0 else -1

colors = plt.cm.tab10(np.linspace(0, 1, 10))
Tb_GHE_monthly = {}
Tf_GHE_monthly = {}
Tb_GHE_hourly_ST = {}
Tf_GHE_hourly_ST = {}

## options without short-term effects
options = {"disp": False, "profiles": True, "method": "equivalent"}
## set options
borefield.set_options_gfunction_calculation(options)

L3 = borefield.size_L3()
print("L3: ", L3)
L4 = borefield.size_L4()
print("L4: ", L4)
print("Yearly injection: ", np.sum(bor_injection_SizeOpt)/1e3, "MWh")
print("Yearly extraction: ", np.sum(bor_extraction_SizeOpt)/1e3, "MWh")
print("Yearly imbalance: ", (np.sum(bor_injection_SizeOpt) - np.sum(bor_extraction_SizeOpt))/1e3, "MWh")
borefield.print_temperature_profile(plot_hourly=True)

In [ ]:
QBor_TACO = df['QBor'].to_numpy()/1e3
QBor_SizeOpt = bor_injection_SizeOpt - bor_extraction_SizeOpt
plt.figure("Compare borefield loads")
plt.plot(QBor_TACO, label="TACO load", color='blue')
plt.plot(QBor_SizeOpt, label="SizeOpt load", color='orange')
plt.xlabel("Time step")
plt.ylabel("Borefield load [kW]")   
plt.legend(), plt.grid()
plt.show()

In [ ]:
# Calculate COPs
TSup = df["T_COP_Sup"].to_numpy()
TBor = df["T_COP_Bor"].to_numpy()
TAir = df["T_COP_Air"].to_numpy()
ASHP_table_path = r"C:\Workdir\Develop\IOCS\Datasheets\HPs\COPTable_VitoCal200A.csv"
GSHP_table_path = r"C:\Workdir\Develop\IOCS\Datasheets\HPs\COPTable_VitoCal300G_BWS_A21.csv"	
cop = COP(COP_ASHP_table_path=ASHP_table_path, COP_GSHP_table_path=GSHP_table_path)
print("Calculating COPs...")
COP_GSHP = np.zeros(len(TSup))
COP_ASHP = np.zeros(len(TSup))

for i in range(len(TSup)):
    COP_GSHP[i] = cop.get_COP_GSHP(Tsink=TSup[i], Tsource=TBor[i])
    COP_ASHP[i] = cop.get_COP_ASHP(Tsink=TSup[i], Tsource=TAir[i])


In [ ]:
COP_TACO = df["COP_GSHP"].to_numpy()
QCon_GSHP_TACO = df["QCon_GSHP"].to_numpy()/1e3
QEva_GSHP_TACO = df["QEva_GSHP"].to_numpy()/1e3
QEva_GSHP_TACO_calc = QCon_GSHP_TACO *(1 - 1/COP_TACO)

print("sum QEva_GSHP_TACO: ", np.sum(QEva_GSHP_TACO)/1e3, "MWh")
print("sum QEva_GSHP_TACO_calc: ", np.sum(QEva_GSHP_TACO_calc)/1e3, "MWh")

In [ ]:
import os
QCon_GSHP_sizeopt = result_dict["GSHP"]["heat"]
QCon_ASHP_sizeopt = result_dict["ASHP"]["heat"]
path_input_data = r"C:\Workdir\Develop\IOCS\input_data"
COP_GSHP_sizeopt = np.loadtxt(os.path.join(path_input_data, "COP_GSHP.txt"))
COP_ASHP_sizeopt = np.loadtxt(os.path.join(path_input_data, "COP_ASHP.txt"))


QCon_GSHP_TACO = df["QCon_GSHP"].to_numpy()/1e3
QCon_ASHP_TACO = df["QCon_ASHP"].to_numpy()/1e3
COP_GSHP_TACO = df["COP_GSHP"].to_numpy()
COP_ASHP_TACO = df["COP_ASHP"].to_numpy()
TSup = df["T_COP_Sup"].to_numpy()
Tair = df["T_COP_Air"].to_numpy()
TBor = df["T_COP_Bor"].to_numpy()


# Calculate daily sums
QCon_ASHP_sizeopt_daily = np.sum(np.array(QCon_ASHP_sizeopt).reshape(-1, 24), axis=1)
QCon_ASHP_TACO_daily = np.sum(QCon_ASHP_TACO.reshape(-1, 24), axis=1)
QCon_GSHP_sizeopt_daily = np.sum(np.array(QCon_GSHP_sizeopt).reshape(-1, 24), axis=1)
QCon_GSHP_TACO_daily = np.sum(QCon_GSHP_TACO.reshape(-1, 24), axis=1)

plt.figure("Compare sizeopt vs TACO")
nb_subplots = 3
ax1 = plt.subplot(nb_subplots, 1, 1)
plt.plot(QCon_GSHP_sizeopt, label="GSHP Sizeopt", color="blue")
plt.plot(QCon_GSHP_TACO, label="GSHP Taco", color="blue", linestyle="--")
plt.plot(QCon_ASHP_sizeopt, label="ASHP Sizeopt", color="orange")
plt.plot(QCon_ASHP_TACO, label="ASHP Taco", color="orange", linestyle="--")
plt.ylabel("QCon [kW]"), plt.title("heating power"), plt.legend(), plt.grid()
ax2 = plt.subplot(nb_subplots, 1, 2, sharex=ax1)
plt.plot(COP_GSHP_sizeopt, label="GSHP Sizeopt", color="blue")
plt.plot(COP_GSHP_TACO, label="GSHP Taco", color="blue", linestyle="--")
plt.plot(COP_ASHP_sizeopt, label="ASHP Sizeopt", color="orange")
plt.plot(COP_ASHP_TACO, label="ASHP Taco", color="orange", linestyle="--")
plt.ylabel("COP"), plt.title("COP"), plt.legend(), plt.grid()
ax3 = plt.subplot(nb_subplots, 1, 3, sharex=ax1)
plt.plot(TSup, label="Taco Sup", color="blue")
plt.plot(Tair, label="Taco Air", color="orange")
plt.plot(TBor, label="Taco Bore", color="green")
plt.ylabel("T [°C]"), plt.title("Temperatures"), plt.legend(), plt.grid()

plt.figure("Daily compare")
nb_subplots = 1
ax1 = plt.subplot(nb_subplots, 1, 1)
plt.plot(QCon_GSHP_sizeopt_daily, label="GSHP Sizeopt", color="blue")
plt.plot(QCon_GSHP_TACO_daily, label="GSHP Taco", color="blue", linestyle="--")
plt.plot(QCon_ASHP_sizeopt_daily, label="ASHP Sizeopt", color="orange")
plt.plot(QCon_ASHP_TACO_daily, label="ASHP Taco", color="orange", linestyle="--")
plt.ylabel("QCon [kW]"), plt.title("heating power"), plt.legend(), plt.grid()



plt.show()

